# Controlled GC Interventions in DNABERT-2

## Objective

Test whether controlled changes in sequence GC content causally alter the promoter-associated representation found in the Layer-1 MLP of baseline DNABERT-2.

## Previous result

Notebook 5 found that GC content was the dominant simple sequence feature associated with Layer-1 MLP transfer strength.

Among unrelated promoter donors:

- GC fraction correlated with MLP rescue;
- the association remained after controlling for donor Layer-5 probe strength and recipient difficulty;
- adding GC fraction improved repeated cross-validated prediction by approximately 0.106 R²;
- additional motif and composition features did not improve generalization beyond GC.

These findings establish association, not causation.

## Main questions

1. Does increasing GC content increase Layer-5 promoter-probe evidence?
2. Does decreasing GC content reduce promoter-probe evidence?
3. Do GC interventions alter Layer-1 MLP activations?
4. Does the change follow GC magnitude monotonically?
5. Can GC effects be separated from generic sequence disruption?
6. Do GC-altered Layer-1 MLP outputs transfer corresponding promoter evidence during patching?

## Experimental design

For each source sequence, construct controlled sequence variants with:

- unchanged sequence length;
- predefined GC changes;
- distributed substitutions rather than one contiguous mutation block;
- preserved bases wherever possible;
- multiple random intervention seeds;
- mutation-count-matched controls that do not systematically change GC.

The experiment will compare:

- original sequence;
- GC-increased variant;
- GC-decreased variant;
- mutation-count-matched neutral control.

## Interpretation boundary

A probe-score change following GC manipulation would show that GC-associated sequence changes causally affect the diagnostic representation.

It would not, by itself, establish that the Layer-1 MLP explicitly calculates GC percentage. GC may still serve as a proxy for another sequence property changed by the intervention.

# Cell 1 — Environment and output setup

In [ ]:
from pathlib import Path
from collections import Counter
import random
import re
import math

import numpy as np
import pandas as pd
import torch
import joblib

import transformers
import tokenizers
import huggingface_hub
import sklearn

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False,
)

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print(
    "Hugging Face Hub:",
    huggingface_hub.__version__,
)
print("Scikit-learn:", sklearn.__version__)
print("CUDA available:", torch.cuda.is_available())


BASE_DIR = Path(
    "/content/drive/MyDrive/algoverse/dnabert"
)

OUTPUT_DIR = BASE_DIR / "outputs"

CAUSAL_DIR = (
    OUTPUT_DIR
    / "promoter_causal_interventions"
)

NOTEBOOK5_DIR = (
    OUTPUT_DIR
    / "promoter_mlp_biological_interpretation"
)

NOTEBOOK6_DIR = (
    OUTPUT_DIR
    / "gc_controlled_interventions"
)

NOTEBOOK6_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROBE_PATH = (
    OUTPUT_DIR
    / "promoter_baseline_pilot"
    / "dnabert_layer5_probe.joblib"
)

PROMOTER_FEATURE_PATH = (
    CAUSAL_DIR
    / "notebook5_matched_promoter_features.csv"
)

NON_PROMOTER_FEATURE_PATH = (
    CAUSAL_DIR
    / "notebook5_non_promoter_features.csv"
)

required_paths = {
    "Layer-5 diagnostic probe": PROBE_PATH,
    "promoter feature table": (
        PROMOTER_FEATURE_PATH
    ),
    "non-promoter feature table": (
        NON_PROMOTER_FEATURE_PATH
    ),
}

print("\nRequired inputs:")

all_inputs_found = True

for name, path in required_paths.items():
    exists = path.exists()

    print(
        f"{name}:",
        "FOUND" if exists else "MISSING",
        "\n ",
        path,
    )

    all_inputs_found &= exists

assert all_inputs_found, (
    "Notebook 6 inputs are missing."
)

print("\nNotebook 6 output directory:")
print(NOTEBOOK6_DIR)

Mounted at /content/drive
Torch: 2.11.0+cu128
Transformers: 5.13.1
Tokenizers: 0.22.2
Hugging Face Hub: 1.23.0
Scikit-learn: 1.6.1
CUDA available: True

Required inputs:
Layer-5 diagnostic probe: FOUND 
  /content/drive/MyDrive/algoverse/dnabert/outputs/promoter_baseline_pilot/dnabert_layer5_probe.joblib
promoter feature table: FOUND 
  /content/drive/MyDrive/algoverse/dnabert/outputs/promoter_causal_interventions/notebook5_matched_promoter_features.csv
non-promoter feature table: FOUND 
  /content/drive/MyDrive/algoverse/dnabert/outputs/promoter_causal_interventions/notebook5_non_promoter_features.csv

Notebook 6 output directory:
/content/drive/MyDrive/algoverse/dnabert/outputs/gc_controlled_interventions


In [ ]:
# Install only the DNABERT-2 compatibility packages.
# Keep scikit-learn 1.6.1 because it matches the saved probe.

!pip install -q --no-cache-dir --force-reinstall \
    transformers==4.44.2 \
    tokenizers==0.19.1 \
    huggingface-hub==0.24.7

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 95.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 286.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 340.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 252.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 142.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 320.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 438.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.6/206.6 kB 451.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 262.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 340.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 445.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.8/801.8 kB 438.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
import torch
import transformers
import tokenizers
import huggingface_hub
import sklearn

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("Scikit-learn:", sklearn.__version__)
print("CUDA available:", torch.cuda.is_available())
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "None",
)

Torch: 2.11.0+cu128
Transformers: 4.44.2
Tokenizers: 0.19.1
Hugging Face Hub: 0.24.7
Scikit-learn: 1.6.1
CUDA available: True
GPU: Tesla T4


# Cell 2 — Load sequences and define controlled GC interventions

In [ ]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

BASE_DIR = Path(
    "/content/drive/MyDrive/algoverse/dnabert"
)

OUTPUT_DIR = BASE_DIR / "outputs"

CAUSAL_DIR = (
    OUTPUT_DIR
    / "promoter_causal_interventions"
)

NOTEBOOK6_DIR = (
    OUTPUT_DIR
    / "gc_controlled_interventions"
)

NOTEBOOK6_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROMOTER_FEATURE_PATH = (
    CAUSAL_DIR
    / "notebook5_matched_promoter_features.csv"
)

NON_PROMOTER_FEATURE_PATH = (
    CAUSAL_DIR
    / "notebook5_non_promoter_features.csv"
)

promoter_feature_df = pd.read_csv(
    PROMOTER_FEATURE_PATH
)

non_promoter_feature_df = pd.read_csv(
    NON_PROMOTER_FEATURE_PATH
)

print("Promoter rows:", len(promoter_feature_df))
print("Non-promoter rows:", len(non_promoter_feature_df))

assert len(promoter_feature_df) == 189
assert len(non_promoter_feature_df) == 189

assert promoter_feature_df[
    "donor_sequence"
].str.len().eq(70).all()

assert non_promoter_feature_df[
    "donor_sequence"
].str.len().eq(70).all()


def gc_fraction(sequence):
    sequence = sequence.upper()

    return (
        sequence.count("G")
        + sequence.count("C")
    ) / len(sequence)


def mutation_positions(
    original_sequence,
    mutated_sequence,
):
    return [
        index
        for index, (
            original_base,
            mutated_base,
        ) in enumerate(
            zip(
                original_sequence,
                mutated_sequence,
            )
        )
        if original_base != mutated_base
    ]


def increase_gc_content(
    sequence,
    number_of_mutations,
    random_generator,
):
    sequence_array = np.array(
        list(sequence.upper())
    )

    candidate_positions = np.where(
        np.isin(
            sequence_array,
            ["A", "T"],
        )
    )[0]

    actual_mutations = min(
        number_of_mutations,
        len(candidate_positions),
    )

    selected_positions = (
        random_generator.choice(
            candidate_positions,
            size=actual_mutations,
            replace=False,
        )
    )

    for position in selected_positions:
        sequence_array[position] = (
            random_generator.choice(
                ["G", "C"]
            )
        )

    return "".join(sequence_array)


def decrease_gc_content(
    sequence,
    number_of_mutations,
    random_generator,
):
    sequence_array = np.array(
        list(sequence.upper())
    )

    candidate_positions = np.where(
        np.isin(
            sequence_array,
            ["G", "C"],
        )
    )[0]

    actual_mutations = min(
        number_of_mutations,
        len(candidate_positions),
    )

    selected_positions = (
        random_generator.choice(
            candidate_positions,
            size=actual_mutations,
            replace=False,
        )
    )

    for position in selected_positions:
        sequence_array[position] = (
            random_generator.choice(
                ["A", "T"]
            )
        )

    return "".join(sequence_array)


def gc_neutral_mutation(
    sequence,
    number_of_mutations,
    random_generator,
):
    sequence_array = np.array(
        list(sequence.upper())
    )

    all_positions = np.arange(
        len(sequence_array)
    )

    actual_mutations = min(
        number_of_mutations,
        len(all_positions),
    )

    selected_positions = (
        random_generator.choice(
            all_positions,
            size=actual_mutations,
            replace=False,
        )
    )

    replacement_map = {
        "A": "T",
        "T": "A",
        "G": "C",
        "C": "G",
    }

    for position in selected_positions:
        sequence_array[position] = (
            replacement_map[
                sequence_array[position]
            ]
        )

    return "".join(sequence_array)


GC_MUTATION_LEVELS = [4, 8]
N_VARIANT_SEEDS = 5


example_sequence = promoter_feature_df[
    "donor_sequence"
].iloc[0]

example_rng = np.random.default_rng(123)

example_gc_up = increase_gc_content(
    example_sequence,
    number_of_mutations=4,
    random_generator=example_rng,
)

example_gc_down = decrease_gc_content(
    example_sequence,
    number_of_mutations=4,
    random_generator=example_rng,
)

example_neutral = gc_neutral_mutation(
    example_sequence,
    number_of_mutations=4,
    random_generator=example_rng,
)

smoke_test_df = pd.DataFrame([
    {
        "variant": "original",
        "gc_fraction": gc_fraction(
            example_sequence
        ),
        "number_of_mutations": 0,
        "sequence": example_sequence,
    },
    {
        "variant": "GC increased",
        "gc_fraction": gc_fraction(
            example_gc_up
        ),
        "number_of_mutations": len(
            mutation_positions(
                example_sequence,
                example_gc_up,
            )
        ),
        "sequence": example_gc_up,
    },
    {
        "variant": "GC decreased",
        "gc_fraction": gc_fraction(
            example_gc_down
        ),
        "number_of_mutations": len(
            mutation_positions(
                example_sequence,
                example_gc_down,
            )
        ),
        "sequence": example_gc_down,
    },
    {
        "variant": "GC neutral",
        "gc_fraction": gc_fraction(
            example_neutral
        ),
        "number_of_mutations": len(
            mutation_positions(
                example_sequence,
                example_neutral,
            )
        ),
        "sequence": example_neutral,
    },
])

display(smoke_test_df)

assert (
    gc_fraction(example_gc_up)
    > gc_fraction(example_sequence)
)

assert (
    gc_fraction(example_gc_down)
    < gc_fraction(example_sequence)
)

assert np.isclose(
    gc_fraction(example_neutral),
    gc_fraction(example_sequence),
)

print("\nControlled GC mutation functions passed.")

Promoter rows: 189
Non-promoter rows: 189


,variant,gc_fraction,number_of_mutations,sequence
0,original,0.828571,0,CCCGCGGAGGAGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...
1,GC increased,0.885714,4,CCCGCGGGGGAGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...
2,GC decreased,0.771429,4,CCCGCGGAGGAGCCCGCGCCCGCCGGGATTTCCAGCCCCGCGCCGC...
3,GC neutral,0.828571,4,CCCGCGGAGGAGCCCGGCCCCGCCGGGATGCCCAGCCCCGCGCCGC...



Controlled GC mutation functions passed.


# Cell 3 — Generate controlled GC intervention variants


In [ ]:
variant_rows = []

source_groups = [
    (
        "promoter",
        promoter_feature_df[
            "donor_sequence"
        ].tolist(),
    ),
    (
        "non-promoter",
        non_promoter_feature_df[
            "donor_sequence"
        ].tolist(),
    ),
]

for source_class, sequences in source_groups:
    for source_index, original_sequence in enumerate(
        sequences
    ):
        original_gc = gc_fraction(
            original_sequence
        )

        for mutation_level in GC_MUTATION_LEVELS:
            for variant_seed in range(
                N_VARIANT_SEEDS
            ):
                # Deterministic RNG specific to this sequence,
                # class, mutation level, and replicate.
                class_offset = (
                    0
                    if source_class == "promoter"
                    else 1_000_000
                )

                local_seed = (
                    SEED
                    + class_offset
                    + source_index * 10_000
                    + mutation_level * 100
                    + variant_seed
                )

                local_rng = np.random.default_rng(
                    local_seed
                )

                gc_up_sequence = increase_gc_content(
                    original_sequence,
                    number_of_mutations=(
                        mutation_level
                    ),
                    random_generator=local_rng,
                )

                gc_down_sequence = decrease_gc_content(
                    original_sequence,
                    number_of_mutations=(
                        mutation_level
                    ),
                    random_generator=local_rng,
                )

                neutral_sequence = gc_neutral_mutation(
                    original_sequence,
                    number_of_mutations=(
                        mutation_level
                    ),
                    random_generator=local_rng,
                )

                variants = {
                    "original": original_sequence,
                    "gc_increased": gc_up_sequence,
                    "gc_decreased": gc_down_sequence,
                    "gc_neutral": neutral_sequence,
                }

                for (
                    intervention_type,
                    variant_sequence,
                ) in variants.items():
                    variant_gc = gc_fraction(
                        variant_sequence
                    )

                    changed_positions = (
                        mutation_positions(
                            original_sequence,
                            variant_sequence,
                        )
                    )

                    variant_rows.append({
                        "source_class": source_class,
                        "source_index": source_index,
                        "mutation_level": (
                            mutation_level
                        ),
                        "variant_seed": variant_seed,
                        "intervention_type": (
                            intervention_type
                        ),
                        "original_sequence": (
                            original_sequence
                        ),
                        "variant_sequence": (
                            variant_sequence
                        ),
                        "original_gc_fraction": (
                            original_gc
                        ),
                        "variant_gc_fraction": (
                            variant_gc
                        ),
                        "gc_change": (
                            variant_gc
                            - original_gc
                        ),
                        "number_of_mutations": len(
                            changed_positions
                        ),
                        "mutation_positions": (
                            ",".join(
                                map(
                                    str,
                                    changed_positions,
                                )
                            )
                        ),
                        "local_seed": local_seed,
                    })


gc_intervention_df = pd.DataFrame(
    variant_rows
)

print(
    "Generated intervention rows:",
    len(gc_intervention_df),
)

display(gc_intervention_df.head(12))


print("\nRows by source class:")
display(
    gc_intervention_df[
        "source_class"
    ]
    .value_counts()
    .rename_axis("source_class")
    .reset_index(name="count")
)


print("\nRows by intervention type:")
display(
    gc_intervention_df[
        "intervention_type"
    ]
    .value_counts()
    .rename_axis("intervention_type")
    .reset_index(name="count")
)


print(
    "\nMean GC change by intervention "
    "and mutation level:"
)

gc_change_summary = (
    gc_intervention_df
    .groupby(
        [
            "intervention_type",
            "mutation_level",
        ]
    )
    .agg(
        number_of_rows=(
            "gc_change",
            "size",
        ),
        mean_gc_change=(
            "gc_change",
            "mean",
        ),
        min_gc_change=(
            "gc_change",
            "min",
        ),
        max_gc_change=(
            "gc_change",
            "max",
        ),
        mean_number_of_mutations=(
            "number_of_mutations",
            "mean",
        ),
    )
    .reset_index()
)

display(gc_change_summary)


# Validation checks.
expected_rows = (
    2
    * 189
    * len(GC_MUTATION_LEVELS)
    * N_VARIANT_SEEDS
    * 4
)

assert len(gc_intervention_df) == expected_rows

assert gc_intervention_df[
    "variant_sequence"
].str.len().eq(70).all()

assert (
    gc_intervention_df.loc[
        gc_intervention_df[
            "intervention_type"
        ] == "original",
        "number_of_mutations",
    ] == 0
).all()

assert np.allclose(
    gc_intervention_df.loc[
        gc_intervention_df[
            "intervention_type"
        ] == "original",
        "gc_change",
    ],
    0,
)

assert np.allclose(
    gc_intervention_df.loc[
        gc_intervention_df[
            "intervention_type"
        ] == "gc_neutral",
        "gc_change",
    ],
    0,
)

assert (
    gc_intervention_df.loc[
        gc_intervention_df[
            "intervention_type"
        ] == "gc_increased",
        "gc_change",
    ] > 0
).all()

assert (
    gc_intervention_df.loc[
        gc_intervention_df[
            "intervention_type"
        ] == "gc_decreased",
        "gc_change",
    ] < 0
).all()


# Check whether any source sequence lacked enough candidate
# bases for the requested mutation count.
mutation_count_diagnostics = (
    gc_intervention_df[
        gc_intervention_df[
            "intervention_type"
        ] != "original"
    ]
    .assign(
        requested_mutations=lambda frame: (
            frame["mutation_level"]
        ),
        shortfall=lambda frame: (
            frame["mutation_level"]
            - frame["number_of_mutations"]
        ),
    )
)

print(
    "\nRows with mutation-count shortfall:",
    int(
        (
            mutation_count_diagnostics[
                "shortfall"
            ] > 0
        ).sum()
    ),
)

display(
    mutation_count_diagnostics[
        mutation_count_diagnostics[
            "shortfall"
        ] > 0
    ].head(20)
)


gc_intervention_df.to_csv(
    NOTEBOOK6_DIR
    / "gc_intervention_variants.csv",
    index=False,
)

gc_change_summary.to_csv(
    NOTEBOOK6_DIR
    / "gc_intervention_generation_summary.csv",
    index=False,
)

print(
    "\nSaved GC intervention variants to:"
)

print(
    NOTEBOOK6_DIR
    / "gc_intervention_variants.csv"
)

Generated intervention rows: 15120


,source_class,source_index,mutation_level,variant_seed,intervention_type,original_sequence,variant_sequence,original_gc_fraction,variant_gc_fraction,gc_change,number_of_mutations,mutation_positions,local_seed
0,promoter,0,4,0,original,CCCGCGGAGGAGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,CCCGCGGAGGAGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,0.828571,0.828571,0.000000,0,,442
1,promoter,0,4,0,gc_increased,CCCGCGGAGGAGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,CCCGCGGAGGAGCCCGCGCCCGCCGGGATGCCCCGCCCCGCGCCGC...,0.828571,0.885714,0.057143,4,"33,50,63,69",442
2,promoter,0,4,0,gc_decreased,CCCGCGGAGGAGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,CCCGCTGAGGAGCCCGCGCCCGACGGGATGCCCAGCCCCGAGCCGC...,0.828571,0.771429,-0.057143,4,"5,22,40,57",442
3,promoter,0,4,0,gc_neutral,CCCGCGGAGGAGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,CCCGCGCAGGTGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,0.828571,0.828571,0.000000,4,"6,10,47,65",442
4,promoter,0,4,1,original,CCCGCGGAGGAGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,CCCGCGGAGGAGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,0.828571,0.828571,0.000000,0,,443
5,promoter,0,4,1,gc_increased,CCCGCGGAGGAGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,CCCGCGGGGGCGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,0.828571,0.885714,0.057143,4,"7,10,48,67",443
6,promoter,0,4,1,gc_decreased,CCCGCGGAGGAGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,CCCGCGGAGGAGCCCGCGCCCGCCGGGATTCCCAGCCCCGAGCCGA...,0.828571,0.771429,-0.057143,4,"29,40,45,55",443
7,promoter,0,4,1,gc_neutral,CCCGCGGAGGAGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,CCCGCGGACGAGCCCGCCCCCGGCGGGATGGCCAGCCCCGCGCCGC...,0.828571,0.828571,0.000000,4,"8,17,22,30",443
8,promoter,0,4,2,original,CCCGCGGAGGAGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,CCCGCGGAGGAGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,0.828571,0.828571,0.000000,0,,444
9,promoter,0,4,2,gc_increased,CCCGCGGAGGAGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,CCCGCGGAGGGGCCCGCGCCCGCCGGGATGCCCAGCCCCGCGCCGC...,0.828571,0.885714,0.057143,4,"10,50,58,67",444



Rows by source class:


,source_class,count
0,promoter,7560
1,non-promoter,7560



Rows by intervention type:


...


Mean GC change by intervention and mutation level:


,intervention_type,mutation_level,number_of_rows,mean_gc_change,min_gc_change,max_gc_change,mean_number_of_mutations
0,gc_decreased,4,1890,-0.057143,-0.057143,-0.057143,4.0
1,gc_decreased,8,1890,-0.114286,-0.114286,-0.114286,8.0
2,gc_increased,4,1890,0.057143,0.057143,0.057143,4.0
3,gc_increased,8,1890,0.114286,0.114286,0.114286,8.0
4,gc_neutral,4,1890,0.000000,0.000000,0.000000,4.0
5,gc_neutral,8,1890,0.000000,0.000000,0.000000,8.0
6,original,4,1890,0.000000,0.000000,0.000000,0.0
7,original,8,1890,0.000000,0.000000,0.000000,0.0



Rows with mutation-count shortfall: 0


,source_class,source_index,mutation_level,variant_seed,intervention_type,original_sequence,variant_sequence,original_gc_fraction,variant_gc_fraction,gc_change,number_of_mutations,mutation_positions,local_seed,requested_mutations,shortfall



Saved GC intervention variants to:
/content/drive/MyDrive/algoverse/dnabert/outputs/gc_controlled_interventions/gc_intervention_variants.csv


# Cell 4 — Load DNABERT-2, tokenizer, and fixed Layer-5 probe

In [7]:
import joblib
import torch
import numpy as np
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModel,
)

from transformers.models.bert.configuration_bert import (
    BertConfig,
)

# Define BASE_DIR and OUTPUT_DIR
from pathlib import Path
BASE_DIR = Path(
    "/content/drive/MyDrive/algoverse/dnabert"
)

OUTPUT_DIR = BASE_DIR / "outputs"

MODEL_NAME = (
    "quietflamingo/dnabert2-no-flashattention"
)

PROBE_PATH = (
    OUTPUT_DIR
    / "promoter_baseline_pilot"
    / "dnabert_layer5_probe.joblib"
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

# Important:
# Load the standard Transformers BertConfig rather than
# the repository's remote custom config class.
config = BertConfig.from_pretrained(
    MODEL_NAME,
)

backbone = AutoModel.from_pretrained(
    MODEL_NAME,
    config=config,
    trust_remote_code=True,
)

backbone.to(device)
backbone.eval()

layer5_probe = joblib.load(
    PROBE_PATH
)

print("Loaded tokenizer:", type(tokenizer))
print("Loaded config:", type(config))
print("Loaded backbone:", type(backbone))
print("Loaded probe:", type(layer5_probe))


# Locate encoder layers.
if hasattr(backbone, "encoder"):
    encoder = backbone.encoder
elif (
    hasattr(backbone, "bert")
    and hasattr(backbone.bert, "encoder")
):
    encoder = backbone.bert.encoder
else:
    raise AttributeError(
        "Could not locate DNABERT-2 encoder."
    )

print(
    "Number of encoder layers:",
    len(encoder.layer),
)

# One-based Layer 5 corresponds to Python index 4.
LAYER5_INDEX = 4

captured_layer5_outputs = []

def layer5_capture_hook(
    module,
    inputs,
    output,
):
    if isinstance(output, tuple):
        hidden_states = output[0]
    else:
        hidden_states = output

    captured_layer5_outputs.append(
        hidden_states.detach()
    )


layer5_hook_handle = (
    encoder.layer[LAYER5_INDEX]
    .register_forward_hook(
        layer5_capture_hook
    )
)


def content_mean_pool(
    hidden_states,
    attention_mask,
):
    pooled_rows = []

    for row_index in range(
        hidden_states.shape[0]
    ):
        valid_length = int(
            attention_mask[row_index]
            .sum()
            .item()
        )

        if valid_length <= 2:
            raise ValueError(
                "Sequence has too few valid tokens."
            )

        content_hidden = hidden_states[
            row_index,
            1:valid_length - 1,
            :,
        ]

        pooled_rows.append(
            content_hidden.mean(dim=0)
        )

    return torch.stack(
        pooled_rows,
        dim=0,
    )


@torch.no_grad()
def extract_layer5_representations(
    sequences,
    batch_size=32,
):
    representations = []

    for batch_start in range(
        0,
        len(sequences),
        batch_size,
    ):
        batch_sequences = sequences[
            batch_start:
            batch_start + batch_size
        ]

        encoded = tokenizer(
            batch_sequences,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )

        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        captured_layer5_outputs.clear()

        _ = backbone(**encoded)

        if len(captured_layer5_outputs) != 1:
            raise RuntimeError(
                "Expected exactly one Layer-5 "
                "hook capture per forward pass."
            )

        layer5_hidden = (
            captured_layer5_outputs[0]
        )

        pooled = content_mean_pool(
            layer5_hidden,
            encoded["attention_mask"],
        )

        representations.append(
            pooled.cpu().numpy()
        )

    return np.concatenate(
        representations,
        axis=0,
    )


def score_sequences_with_probe(
    sequences,
    batch_size=32,
):
    representations = (
        extract_layer5_representations(
            sequences,
            batch_size=batch_size,
        )
    )

    logits = layer5_probe.decision_function(
        representations
    )

    probabilities = (
        layer5_probe.predict_proba(
            representations
        )[:, 1]
    )

    return (
        representations,
        logits,
        probabilities,
    )


# Balanced smoke-test batch.
smoke_sequences = (
    promoter_feature_df[
        "donor_sequence"
    ].head(4).tolist()
    +
    non_promoter_feature_df[
        "donor_sequence"
    ].head(4).tolist()
)

# [staged fix] smoke test uses the 3-D pooler; cell 12 redefines it for DNABERT-2's flattened 2-D output and reruns this.
try:
    (
        smoke_representations,
        smoke_logits,
        smoke_probabilities,
    ) = score_sequences_with_probe(
        smoke_sequences,
        batch_size=8,
    )

    smoke_score_df = pd.DataFrame({
        "expected_class": (
            ["promoter"] * 4
            + ["non-promoter"] * 4
        ),
        "probe_logit": smoke_logits,
        "probe_probability": (
            smoke_probabilities
        ),
    })

    display(smoke_score_df)

    print(
        "\nRepresentation shape:",
        smoke_representations.shape,
    )

    assert (
        smoke_representations.shape[0]
        == 8
    )

    assert np.isfinite(
        smoke_logits
    ).all()

    print(
        "\nDNABERT-2 Layer-5 scoring smoke test passed."
    )
except Exception as exc:
    print("[staged fix] superseded step did not run:", type(exc).__name__, exc)
    print("[staged fix] the following cell redoes this correctly.")


Device: cuda


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/root/.cache/huggingface/modules/transformers_modules/quietflamingo/dnabert2-no-flashattention/813031b2bf86d9e960a027e2734c908009f31601/bert_layers.py:123: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at quietflamingo/dnabert2-no-flashattention and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predi

Loaded tokenizer: <class 'transformers.tokenization_utils_fast.PreTrainedTokenizerFast'>
Loaded config: <class 'transformers.models.bert.configuration_bert.BertConfig'>
Loaded backbone: <class 'transformers_modules.quietflamingo.dnabert2-no-flashattention.813031b2bf86d9e960a027e2734c908009f31601.bert_layers.BertModel'>
Loaded probe: <class 'sklearn.pipeline.Pipeline'>
Number of encoder layers: 12
[staged fix] superseded step did not run: IndexError too many indices for tensor of dimension 2
[staged fix] the following cell redoes this correctly.


In [10]:
# Cell 4B — Handle flattened unpadded DNABERT-2 activations

def split_and_content_mean_pool(
    hidden_states,
    attention_mask,
):
    """
    Handle either:

    1. ordinary padded activations:
       [batch, padded_tokens, hidden]

    2. DNABERT-2 flattened unpadded activations:
       [sum(valid_tokens), hidden]

    The first and final valid tokens are treated as
    special tokens and excluded from pooling.
    """

    valid_lengths = (
        attention_mask
        .sum(dim=1)
        .to(dtype=torch.long)
        .tolist()
    )

    pooled_rows = []

    if hidden_states.ndim == 3:
        # Standard padded representation.
        for row_index, valid_length in enumerate(
            valid_lengths
        ):
            if valid_length <= 2:
                raise ValueError(
                    "Sequence has too few valid tokens."
                )

            content_hidden = hidden_states[
                row_index,
                1:valid_length - 1,
                :,
            ]

            pooled_rows.append(
                content_hidden.mean(dim=0)
            )

    elif hidden_states.ndim == 2:
        # Flattened unpadded DNABERT-2 representation.
        expected_total_tokens = sum(
            valid_lengths
        )

        if (
            hidden_states.shape[0]
            != expected_total_tokens
        ):
            raise RuntimeError(
                "Flattened activation length does not "
                "match the sum of attention-mask lengths. "
                f"Activation tokens: {hidden_states.shape[0]}, "
                f"expected: {expected_total_tokens}."
            )

        token_offset = 0

        for valid_length in valid_lengths:
            if valid_length <= 2:
                raise ValueError(
                    "Sequence has too few valid tokens."
                )

            sequence_hidden = hidden_states[
                token_offset:
                token_offset + valid_length,
                :,
            ]

            content_hidden = sequence_hidden[
                1:valid_length - 1,
                :,
            ]

            pooled_rows.append(
                content_hidden.mean(dim=0)
            )

            token_offset += valid_length

        if token_offset != hidden_states.shape[0]:
            raise RuntimeError(
                "Did not consume all flattened tokens."
            )

    else:
        raise ValueError(
            "Unexpected Layer-5 activation shape: "
            f"{tuple(hidden_states.shape)}"
        )

    return torch.stack(
        pooled_rows,
        dim=0,
    )


@torch.no_grad()
def extract_layer5_representations(
    sequences,
    batch_size=32,
):
    representations = []

    for batch_start in range(
        0,
        len(sequences),
        batch_size,
    ):
        batch_sequences = sequences[
            batch_start:
            batch_start + batch_size
        ]

        encoded = tokenizer(
            batch_sequences,
            padding=True,
            truncation=False,
            return_tensors="pt",
        )

        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        captured_layer5_outputs.clear()

        _ = backbone(**encoded)

        if len(captured_layer5_outputs) != 1:
            raise RuntimeError(
                "Expected exactly one Layer-5 "
                "hook capture per forward pass, "
                f"but received "
                f"{len(captured_layer5_outputs)}."
            )

        layer5_hidden = (
            captured_layer5_outputs[0]
        )

        pooled = split_and_content_mean_pool(
            layer5_hidden,
            encoded["attention_mask"],
        )

        representations.append(
            pooled.cpu().numpy()
        )

    return np.concatenate(
        representations,
        axis=0,
    )


def score_sequences_with_probe(
    sequences,
    batch_size=32,
):
    representations = (
        extract_layer5_representations(
            sequences,
            batch_size=batch_size,
        )
    )

    logits = layer5_probe.decision_function(
        representations
    )

    probabilities = (
        layer5_probe.predict_proba(
            representations
        )[:, 1]
    )

    return (
        representations,
        logits,
        probabilities,
    )


# Rerun the balanced smoke test.
smoke_sequences = (
    promoter_feature_df[
        "donor_sequence"
    ].head(4).tolist()
    +
    non_promoter_feature_df[
        "donor_sequence"
    ].head(4).tolist()
)

(
    smoke_representations,
    smoke_logits,
    smoke_probabilities,
) = score_sequences_with_probe(
    smoke_sequences,
    batch_size=8,
)

smoke_score_df = pd.DataFrame({
    "expected_class": (
        ["promoter"] * 4
        + ["non-promoter"] * 4
    ),
    "probe_logit": smoke_logits,
    "probe_probability": (
        smoke_probabilities
    ),
})

display(smoke_score_df)

print(
    "\nRepresentation shape:",
    smoke_representations.shape,
)

print(
    "Captured raw Layer-5 shape:",
    tuple(
        captured_layer5_outputs[0].shape
    ),
)

print(
    "Attention-mask valid lengths:",
    tokenizer(
        smoke_sequences,
        padding=True,
        truncation=False,
        return_tensors="pt",
    )["attention_mask"].sum(dim=1).tolist(),
)

assert (
    smoke_representations.shape[0]
    == 8
)

assert (
    smoke_representations.shape[1]
    == layer5_probe.n_features_in_
)

assert np.isfinite(
    smoke_logits
).all()

print(
    "\nDNABERT-2 Layer-5 scoring smoke test passed."
)

,expected_class,probe_logit,probe_probability
0,promoter,5.393588,0.995475
1,promoter,5.007419,0.993356
2,promoter,6.289934,0.998149
3,promoter,4.211720,0.985396
4,non-promoter,2.096604,0.890573
5,non-promoter,-6.911240,0.000996
6,non-promoter,1.905344,0.870495
7,non-promoter,-8.689644,0.000168



Representation shape: (8, 768)
Captured raw Layer-5 shape: (142, 768)
Attention-mask valid lengths: [18, 19, 17, 17, 18, 17, 17, 19]

DNABERT-2 Layer-5 scoring smoke test passed.


# Cell 5 — Score all unique GC intervention sequences

In [11]:
from tqdm.auto import tqdm

# Score only unique sequences to avoid redundant model passes.
unique_sequences = (
    gc_intervention_df[
        "variant_sequence"
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "Total intervention rows:",
    len(gc_intervention_df),
)

print(
    "Unique sequences to score:",
    len(unique_sequences),
)

(
    unique_layer5_representations,
    unique_probe_logits,
    unique_probe_probabilities,
) = score_sequences_with_probe(
    unique_sequences.tolist(),
    batch_size=64,
)

print(
    "\nUnique representation matrix:",
    unique_layer5_representations.shape,
)

assert (
    unique_layer5_representations.shape[0]
    == len(unique_sequences)
)

assert (
    unique_layer5_representations.shape[1]
    == 768
)

assert np.isfinite(
    unique_probe_logits
).all()

assert np.isfinite(
    unique_probe_probabilities
).all()


sequence_score_lookup = pd.DataFrame({
    "variant_sequence": unique_sequences,
    "probe_logit": unique_probe_logits,
    "probe_probability": (
        unique_probe_probabilities
    ),
})


gc_intervention_scored_df = (
    gc_intervention_df
    .merge(
        sequence_score_lookup,
        on="variant_sequence",
        how="left",
        validate="many_to_one",
    )
)

assert (
    gc_intervention_scored_df[
        "probe_logit"
    ]
    .notna()
    .all()
)


# Obtain the original score within each matched intervention block.
block_keys = [
    "source_class",
    "source_index",
    "mutation_level",
    "variant_seed",
]

original_score_df = (
    gc_intervention_scored_df[
        gc_intervention_scored_df[
            "intervention_type"
        ] == "original"
    ][
        block_keys
        + [
            "probe_logit",
            "probe_probability",
        ]
    ]
    .rename(
        columns={
            "probe_logit": (
                "original_probe_logit"
            ),
            "probe_probability": (
                "original_probe_probability"
            ),
        }
    )
)


gc_intervention_scored_df = (
    gc_intervention_scored_df
    .merge(
        original_score_df,
        on=block_keys,
        how="left",
        validate="many_to_one",
    )
)


gc_intervention_scored_df[
    "probe_logit_change"
] = (
    gc_intervention_scored_df[
        "probe_logit"
    ]
    - gc_intervention_scored_df[
        "original_probe_logit"
    ]
)

gc_intervention_scored_df[
    "probe_probability_change"
] = (
    gc_intervention_scored_df[
        "probe_probability"
    ]
    - gc_intervention_scored_df[
        "original_probe_probability"
    ]
)


print("\nScored intervention preview:")

display(
    gc_intervention_scored_df[
        [
            "source_class",
            "source_index",
            "mutation_level",
            "variant_seed",
            "intervention_type",
            "original_gc_fraction",
            "variant_gc_fraction",
            "gc_change",
            "probe_logit",
            "original_probe_logit",
            "probe_logit_change",
        ]
    ].head(16)
)


# Confirm repeated originals reproduce exactly.
original_change_max = (
    gc_intervention_scored_df.loc[
        gc_intervention_scored_df[
            "intervention_type"
        ] == "original",
        "probe_logit_change",
    ]
    .abs()
    .max()
)

print(
    "\nMaximum original self-change:",
    original_change_max,
)

assert original_change_max < 1e-8


intervention_score_summary = (
    gc_intervention_scored_df
    .groupby(
        [
            "source_class",
            "mutation_level",
            "intervention_type",
        ]
    )
    .agg(
        number_of_rows=(
            "probe_logit_change",
            "size",
        ),
        mean_gc_change=(
            "gc_change",
            "mean",
        ),
        mean_original_logit=(
            "original_probe_logit",
            "mean",
        ),
        mean_variant_logit=(
            "probe_logit",
            "mean",
        ),
        mean_logit_change=(
            "probe_logit_change",
            "mean",
        ),
        median_logit_change=(
            "probe_logit_change",
            "median",
        ),
        std_logit_change=(
            "probe_logit_change",
            "std",
        ),
        fraction_positive_change=(
            "probe_logit_change",
            lambda values: np.mean(
                values > 0
            ),
        ),
    )
    .reset_index()
)

print(
    "\nInitial intervention score summary:"
)

display(intervention_score_summary)


gc_intervention_scored_df.to_csv(
    NOTEBOOK6_DIR
    / "gc_intervention_scored_results.csv",
    index=False,
)

sequence_score_lookup.to_csv(
    NOTEBOOK6_DIR
    / "gc_unique_sequence_probe_scores.csv",
    index=False,
)

np.savez_compressed(
    NOTEBOOK6_DIR
    / "gc_unique_sequence_layer5_representations.npz",
    representations=(
        unique_layer5_representations
    ),
    logits=unique_probe_logits,
    probabilities=(
        unique_probe_probabilities
    ),
)

intervention_score_summary.to_csv(
    NOTEBOOK6_DIR
    / "gc_intervention_initial_score_summary.csv",
    index=False,
)

print(
    "\nSaved full GC intervention scores to:"
)

print(
    NOTEBOOK6_DIR
    / "gc_intervention_scored_results.csv"
)

Total intervention rows: 15120
Unique sequences to score: 11718

Unique representation matrix: (11718, 768)

Scored intervention preview:


,source_class,source_index,mutation_level,variant_seed,intervention_type,original_gc_fraction,variant_gc_fraction,gc_change,probe_logit,original_probe_logit,probe_logit_change
0,promoter,0,4,0,original,0.828571,0.828571,0.000000,5.393593,5.393593,0.000000
1,promoter,0,4,0,gc_increased,0.828571,0.885714,0.057143,3.231813,5.393593,-2.161781
2,promoter,0,4,0,gc_decreased,0.828571,0.771429,-0.057143,-2.202195,5.393593,-7.595788
3,promoter,0,4,0,gc_neutral,0.828571,0.828571,0.000000,4.870756,5.393593,-0.522837
4,promoter,0,4,1,original,0.828571,0.828571,0.000000,5.393593,5.393593,0.000000
5,promoter,0,4,1,gc_increased,0.828571,0.885714,0.057143,15.060440,5.393593,9.666847
6,promoter,0,4,1,gc_decreased,0.828571,0.771429,-0.057143,6.194853,5.393593,0.801259
7,promoter,0,4,1,gc_neutral,0.828571,0.828571,0.000000,-0.007576,5.393593,-5.401169
8,promoter,0,4,2,original,0.828571,0.828571,0.000000,5.393593,5.393593,0.000000
9,promoter,0,4,2,gc_increased,0.828571,0.885714,0.057143,12.894301,5.393593,7.500707



Maximum original self-change: 0.0

Initial intervention score summary:


,source_class,mutation_level,intervention_type,number_of_rows,mean_gc_change,mean_original_logit,mean_variant_logit,mean_logit_change,median_logit_change,std_logit_change,fraction_positive_change
0,non-promoter,4,gc_decreased,945,-0.057143,-3.255195,-4.134969,-0.879773,-0.941663,3.759795,0.386243
1,non-promoter,4,gc_increased,945,0.057143,-3.255195,-2.117594,1.137601,1.034318,3.628734,0.628571
2,non-promoter,4,gc_neutral,945,0.000000,-3.255195,-3.548135,-0.292939,-0.197306,3.570420,0.479365
3,non-promoter,4,original,945,0.000000,-3.255195,-3.255195,0.000000,0.000000,0.000000,0.000000
4,non-promoter,8,gc_decreased,945,-0.114286,-3.255195,-4.229005,-0.973810,-0.812447,4.572446,0.424339
5,non-promoter,8,gc_increased,945,0.114286,-3.255195,-0.743794,2.511401,2.605885,5.010681,0.712169
6,non-promoter,8,gc_neutral,945,0.000000,-3.255195,-3.611165,-0.355970,-0.454669,4.542565,0.463492
7,non-promoter,8,original,945,0.000000,-3.255195,-3.255195,0.000000,0.000000,0.000000,0.000000
8,promoter,4,gc_decreased,945,-0.057143,7.393424,3.553091,-3.840334,-3.651523,4.348892,0.174603
9,promoter,4,gc_increased,945,0.057143,7.393424,7.111499,-0.281925,-0.139731,4.455525,0.484656



Saved full GC intervention scores to:
/content/drive/MyDrive/algoverse/dnabert/outputs/gc_controlled_interventions/gc_intervention_scored_results.csv


# Cell 6 — Estimate GC-specific effects relative to matched neutral controls

In [12]:
analysis_rows = (
    gc_intervention_scored_df[
        gc_intervention_scored_df[
            "intervention_type"
        ].isin(
            [
                "gc_increased",
                "gc_decreased",
                "gc_neutral",
            ]
        )
    ]
    .copy()
)

block_keys = [
    "source_class",
    "source_index",
    "mutation_level",
    "variant_seed",
]

# Put the three intervention conditions onto the same row.
matched_effect_df = (
    analysis_rows
    .pivot(
        index=block_keys,
        columns="intervention_type",
        values="probe_logit_change",
    )
    .reset_index()
)

matched_effect_df.columns.name = None

required_condition_columns = [
    "gc_increased",
    "gc_decreased",
    "gc_neutral",
]

assert matched_effect_df[
    required_condition_columns
].notna().all().all()


# GC-specific contrasts relative to an equal-mutation neutral control.
matched_effect_df[
    "gc_increase_vs_neutral"
] = (
    matched_effect_df["gc_increased"]
    - matched_effect_df["gc_neutral"]
)

matched_effect_df[
    "gc_decrease_vs_neutral"
] = (
    matched_effect_df["gc_decreased"]
    - matched_effect_df["gc_neutral"]
)

# Direct directional contrast.
matched_effect_df[
    "gc_increase_vs_decrease"
] = (
    matched_effect_df["gc_increased"]
    - matched_effect_df["gc_decreased"]
)


contrast_columns = [
    "gc_increase_vs_neutral",
    "gc_decrease_vs_neutral",
    "gc_increase_vs_decrease",
]


contrast_summary = (
    matched_effect_df
    .groupby(
        [
            "source_class",
            "mutation_level",
        ]
    )
    .agg(
        number_of_blocks=(
            "source_index",
            "size",
        ),
        mean_gc_increase_vs_neutral=(
            "gc_increase_vs_neutral",
            "mean",
        ),
        median_gc_increase_vs_neutral=(
            "gc_increase_vs_neutral",
            "median",
        ),
        fraction_gc_increase_above_neutral=(
            "gc_increase_vs_neutral",
            lambda values: np.mean(values > 0),
        ),
        mean_gc_decrease_vs_neutral=(
            "gc_decrease_vs_neutral",
            "mean",
        ),
        median_gc_decrease_vs_neutral=(
            "gc_decrease_vs_neutral",
            "median",
        ),
        fraction_gc_decrease_below_neutral=(
            "gc_decrease_vs_neutral",
            lambda values: np.mean(values < 0),
        ),
        mean_gc_increase_vs_decrease=(
            "gc_increase_vs_decrease",
            "mean",
        ),
        median_gc_increase_vs_decrease=(
            "gc_increase_vs_decrease",
            "median",
        ),
        fraction_gc_increase_above_decrease=(
            "gc_increase_vs_decrease",
            lambda values: np.mean(values > 0),
        ),
    )
    .reset_index()
)

print("Matched GC-specific contrast summary:")
display(contrast_summary)


# First average the five random seeds for each source sequence.
source_level_effect_df = (
    matched_effect_df
    .groupby(
        [
            "source_class",
            "source_index",
            "mutation_level",
        ]
    )[contrast_columns]
    .mean()
    .reset_index()
)

print("\nSource-level effects after averaging five seeds:")
display(source_level_effect_df.head(12))

assert (
    source_level_effect_df
    .groupby(
        [
            "source_class",
            "mutation_level",
        ]
    )
    .size()
    .eq(189)
    .all()
)


# Cluster bootstrap over source sequences.
N_BOOTSTRAPS = 5000
bootstrap_rng = np.random.default_rng(42)

bootstrap_rows = []

for (
    source_class,
    mutation_level,
), group_df in source_level_effect_df.groupby(
    [
        "source_class",
        "mutation_level",
    ]
):
    for contrast in contrast_columns:
        values = group_df[
            contrast
        ].to_numpy(dtype=float)

        bootstrap_means = np.empty(
            N_BOOTSTRAPS
        )

        for bootstrap_index in range(
            N_BOOTSTRAPS
        ):
            sampled_indices = (
                bootstrap_rng.integers(
                    0,
                    len(values),
                    size=len(values),
                )
            )

            bootstrap_means[
                bootstrap_index
            ] = values[
                sampled_indices
            ].mean()

        bootstrap_rows.append({
            "source_class": source_class,
            "mutation_level": mutation_level,
            "contrast": contrast,
            "number_of_sources": len(values),
            "mean_effect": values.mean(),
            "median_effect": np.median(values),
            "ci_2.5": np.percentile(
                bootstrap_means,
                2.5,
            ),
            "ci_97.5": np.percentile(
                bootstrap_means,
                97.5,
            ),
            "fraction_sources_positive": np.mean(
                values > 0
            ),
            "fraction_sources_negative": np.mean(
                values < 0
            ),
        })


cluster_bootstrap_summary = pd.DataFrame(
    bootstrap_rows
)

print(
    "\nCluster-bootstrap confidence intervals "
    "over source sequences:"
)

display(
    cluster_bootstrap_summary.sort_values(
        [
            "source_class",
            "mutation_level",
            "contrast",
        ]
    )
)


# Test dose scaling from four to eight mutations.
dose_effect_df = (
    source_level_effect_df
    .pivot(
        index=[
            "source_class",
            "source_index",
        ],
        columns="mutation_level",
        values=contrast_columns,
    )
)

dose_effect_rows = []

for source_class in [
    "promoter",
    "non-promoter",
]:
    class_df = dose_effect_df.loc[
        source_class
    ]

    for contrast in contrast_columns:
        four_mutation_effect = (
            class_df[(contrast, 4)]
            .to_numpy()
        )

        eight_mutation_effect = (
            class_df[(contrast, 8)]
            .to_numpy()
        )

        dose_difference = (
            eight_mutation_effect
            - four_mutation_effect
        )

        dose_effect_rows.append({
            "source_class": source_class,
            "contrast": contrast,
            "mean_effect_4_mutations": (
                four_mutation_effect.mean()
            ),
            "mean_effect_8_mutations": (
                eight_mutation_effect.mean()
            ),
            "mean_8_minus_4": (
                dose_difference.mean()
            ),
            "median_8_minus_4": (
                np.median(dose_difference)
            ),
            "fraction_larger_at_8": np.mean(
                eight_mutation_effect
                > four_mutation_effect
            ),
        })


dose_effect_summary = pd.DataFrame(
    dose_effect_rows
)

print("\nDose-response summary:")
display(dose_effect_summary)


matched_effect_df.to_csv(
    NOTEBOOK6_DIR
    / "gc_matched_neutral_contrasts.csv",
    index=False,
)

source_level_effect_df.to_csv(
    NOTEBOOK6_DIR
    / "gc_source_level_contrasts.csv",
    index=False,
)

cluster_bootstrap_summary.to_csv(
    NOTEBOOK6_DIR
    / "gc_source_cluster_bootstrap_summary.csv",
    index=False,
)

dose_effect_summary.to_csv(
    NOTEBOOK6_DIR
    / "gc_dose_response_summary.csv",
    index=False,
)

print(
    "\nSaved GC-specific contrast analyses to:",
    NOTEBOOK6_DIR,
)

Matched GC-specific contrast summary:


,source_class,mutation_level,number_of_blocks,mean_gc_increase_vs_neutral,median_gc_increase_vs_neutral,fraction_gc_increase_above_neutral,mean_gc_decrease_vs_neutral,median_gc_decrease_vs_neutral,fraction_gc_decrease_below_neutral,mean_gc_increase_vs_decrease,median_gc_increase_vs_decrease,fraction_gc_increase_above_decrease
0,non-promoter,4,945,1.430540,1.218105,0.619048,-0.586834,-0.687871,0.564021,2.017375,2.117836,0.673016
1,non-promoter,8,945,2.867371,2.827646,0.683598,-0.617840,-0.730761,0.538624,3.485211,3.310728,0.707937
2,promoter,4,945,2.039341,1.985369,0.648677,-1.519068,-1.587926,0.616931,3.558409,3.526349,0.730159
3,promoter,8,945,4.979675,4.875183,0.771429,-2.510489,-2.479217,0.649735,7.490164,7.428267,0.817989



Source-level effects after averaging five seeds:


,source_class,source_index,mutation_level,gc_increase_vs_neutral,gc_decrease_vs_neutral,gc_increase_vs_decrease
0,non-promoter,0,4,0.844857,-1.269654,2.114511
1,non-promoter,0,8,2.387096,2.361039,0.026056
2,non-promoter,1,4,1.986610,-1.113879,3.100489
3,non-promoter,1,8,5.141372,-0.969648,6.111020
4,non-promoter,2,4,2.263915,-0.085906,2.349822
5,non-promoter,2,8,3.038030,0.312101,2.725929
6,non-promoter,3,4,0.538208,-2.997455,3.535662
7,non-promoter,3,8,7.619440,-4.279417,11.898857
8,non-promoter,4,4,-1.342887,-0.294190,-1.048696
9,non-promoter,4,8,4.805624,-1.613889,6.419513



Cluster-bootstrap confidence intervals over source sequences:


,source_class,mutation_level,contrast,number_of_sources,mean_effect,median_effect,ci_2.5,ci_97.5,fraction_sources_positive,fraction_sources_negative
1,non-promoter,4,gc_decrease_vs_neutral,189,-0.586834,-0.453092,-0.914678,-0.267577,0.433862,0.566138
2,non-promoter,4,gc_increase_vs_decrease,189,2.017375,1.857662,1.600178,2.427102,0.751323,0.248677
0,non-promoter,4,gc_increase_vs_neutral,189,1.430540,1.161328,1.085365,1.767287,0.714286,0.285714
4,non-promoter,8,gc_decrease_vs_neutral,189,-0.617840,-0.542536,-1.017824,-0.211525,0.428571,0.571429
5,non-promoter,8,gc_increase_vs_decrease,189,3.485211,3.315811,2.912908,4.040859,0.825397,0.174603
3,non-promoter,8,gc_increase_vs_neutral,189,2.867371,2.821166,2.422488,3.315170,0.793651,0.206349
7,promoter,4,gc_decrease_vs_neutral,189,-1.519068,-1.769833,-1.910755,-1.145499,0.253968,0.746032
8,promoter,4,gc_increase_vs_decrease,189,3.558409,3.525376,2.969742,4.135356,0.820106,0.179894
6,promoter,4,gc_increase_vs_neutral,189,2.039341,1.955535,1.613880,2.473293,0.756614,0.243386
10,promoter,8,gc_decrease_vs_neutral,189,-2.510489,-2.608828,-3.036433,-1.963234,0.264550,0.735450



Dose-response summary:


,source_class,contrast,mean_effect_4_mutations,mean_effect_8_mutations,mean_8_minus_4,median_8_minus_4,fraction_larger_at_8
0,promoter,gc_increase_vs_neutral,2.039341,4.979675,2.940334,3.074595,0.777778
1,promoter,gc_decrease_vs_neutral,-1.519068,-2.510489,-0.991421,-1.253342,0.391534
2,promoter,gc_increase_vs_decrease,3.558409,7.490164,3.931755,4.058147,0.830688
3,non-promoter,gc_increase_vs_neutral,1.430540,2.867371,1.436830,1.636443,0.666667
4,non-promoter,gc_decrease_vs_neutral,-0.586834,-0.617840,-0.031006,0.237272,0.534392
5,non-promoter,gc_increase_vs_decrease,2.017375,3.485211,1.467836,1.399517,0.682540



Saved GC-specific contrast analyses to: /content/drive/MyDrive/algoverse/dnabert/outputs/gc_controlled_interventions


# Cell 7 — Capture pooled Layer-1 MLP outputs for all unique variants

In [13]:
captured_layer1_mlp_outputs = []


def layer1_mlp_capture_hook(
    module,
    inputs,
    output,
):
    if isinstance(output, tuple):
        branch_output = output[0]
    else:
        branch_output = output

    captured_layer1_mlp_outputs.append(
        branch_output.detach()
    )


# Layer 1 in one-based notation is encoder.layer[0].
LAYER1_INDEX = 0

layer1_mlp_module = (
    encoder.layer[LAYER1_INDEX]
    .mlp.wo
)

layer1_mlp_hook_handle = (
    layer1_mlp_module.register_forward_hook(
        layer1_mlp_capture_hook
    )
)

print(
    "Layer-1 MLP module:",
    layer1_mlp_module,
)


@torch.no_grad()
def extract_layer1_mlp_representations(
    sequences,
    batch_size=64,
):
    pooled_representations = []

    for batch_start in range(
        0,
        len(sequences),
        batch_size,
    ):
        batch_sequences = sequences[
            batch_start:
            batch_start + batch_size
        ]

        encoded = tokenizer(
            batch_sequences,
            padding=True,
            truncation=False,
            return_tensors="pt",
        )

        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        captured_layer1_mlp_outputs.clear()

        _ = backbone(**encoded)

        if len(captured_layer1_mlp_outputs) != 1:
            raise RuntimeError(
                "Expected exactly one Layer-1 MLP capture, "
                f"received {len(captured_layer1_mlp_outputs)}."
            )

        layer1_mlp_output = (
            captured_layer1_mlp_outputs[0]
        )

        pooled = split_and_content_mean_pool(
            layer1_mlp_output,
            encoded["attention_mask"],
        )

        pooled_representations.append(
            pooled.cpu().numpy()
        )

    return np.concatenate(
        pooled_representations,
        axis=0,
    )


# First run a small smoke test.
layer1_mlp_smoke = (
    extract_layer1_mlp_representations(
        smoke_sequences,
        batch_size=8,
    )
)

print(
    "Layer-1 MLP smoke-test shape:",
    layer1_mlp_smoke.shape,
)

print(
    "Finite smoke-test values:",
    np.isfinite(layer1_mlp_smoke).all(),
)

assert layer1_mlp_smoke.shape == (8, 768)
assert np.isfinite(layer1_mlp_smoke).all()


# Extract representations for the same unique-sequence ordering
# used in Cell 5.
unique_layer1_mlp_representations = (
    extract_layer1_mlp_representations(
        unique_sequences.tolist(),
        batch_size=64,
    )
)

print(
    "\nFull Layer-1 MLP representation matrix:",
    unique_layer1_mlp_representations.shape,
)

assert (
    unique_layer1_mlp_representations.shape
    == (
        len(unique_sequences),
        768,
    )
)

assert np.isfinite(
    unique_layer1_mlp_representations
).all()


# Preserve the exact row mapping.
layer1_mlp_sequence_index_df = pd.DataFrame({
    "representation_index": np.arange(
        len(unique_sequences)
    ),
    "variant_sequence": unique_sequences,
    "probe_logit": unique_probe_logits,
    "probe_probability": (
        unique_probe_probabilities
    ),
})


np.savez_compressed(
    NOTEBOOK6_DIR
    / "gc_unique_sequence_layer1_mlp_representations.npz",
    representations=(
        unique_layer1_mlp_representations
    ),
)

layer1_mlp_sequence_index_df.to_csv(
    NOTEBOOK6_DIR
    / "gc_unique_sequence_layer1_mlp_index.csv",
    index=False,
)

print(
    "\nSaved Layer-1 MLP representations to:"
)

print(
    NOTEBOOK6_DIR
    / "gc_unique_sequence_layer1_mlp_representations.npz"
)

print(
    "\nSaved representation index to:"
)

print(
    NOTEBOOK6_DIR
    / "gc_unique_sequence_layer1_mlp_index.csv"
)

Layer-1 MLP module: Linear(in_features=3072, out_features=768, bias=True)
Layer-1 MLP smoke-test shape: (8, 768)
Finite smoke-test values: True

Full Layer-1 MLP representation matrix: (11718, 768)

Saved Layer-1 MLP representations to:
/content/drive/MyDrive/algoverse/dnabert/outputs/gc_controlled_interventions/gc_unique_sequence_layer1_mlp_representations.npz

Saved representation index to:
/content/drive/MyDrive/algoverse/dnabert/outputs/gc_controlled_interventions/gc_unique_sequence_layer1_mlp_index.csv


# Cell 8 — Build a promoter direction in Layer-1 MLP space
# and test whether GC interventions move along it


In [14]:
# Map every unique sequence to its Layer-1 MLP representation.
sequence_to_representation_index = dict(
    zip(
        layer1_mlp_sequence_index_df[
            "variant_sequence"
        ],
        layer1_mlp_sequence_index_df[
            "representation_index"
        ],
    )
)


def get_mlp_representations_for_sequences(
    sequences,
):
    indices = [
        sequence_to_representation_index[
            sequence
        ]
        for sequence in sequences
    ]

    return unique_layer1_mlp_representations[
        indices
    ]


# Use each original sequence only once.
original_promoter_sequences = (
    promoter_feature_df[
        "donor_sequence"
    ].tolist()
)

original_non_promoter_sequences = (
    non_promoter_feature_df[
        "donor_sequence"
    ].tolist()
)

original_promoter_mlp = (
    get_mlp_representations_for_sequences(
        original_promoter_sequences
    )
)

original_non_promoter_mlp = (
    get_mlp_representations_for_sequences(
        original_non_promoter_sequences
    )
)

print(
    "Original promoter MLP matrix:",
    original_promoter_mlp.shape,
)

print(
    "Original non-promoter MLP matrix:",
    original_non_promoter_mlp.shape,
)


# Difference-of-means promoter direction.
promoter_mean_vector = (
    original_promoter_mlp.mean(axis=0)
)

non_promoter_mean_vector = (
    original_non_promoter_mlp.mean(axis=0)
)

promoter_direction = (
    promoter_mean_vector
    - non_promoter_mean_vector
)

promoter_direction_norm = np.linalg.norm(
    promoter_direction
)

promoter_direction_unit = (
    promoter_direction
    / promoter_direction_norm
)

print(
    "\nPromoter direction norm:",
    promoter_direction_norm,
)

assert np.isclose(
    np.linalg.norm(promoter_direction_unit),
    1.0,
)


def project_onto_promoter_direction(
    representations,
):
    return (
        representations
        @ promoter_direction_unit
    )


# Validate that original promoter sequences score higher
# than original non-promoters on this direction.
promoter_direction_scores = (
    project_onto_promoter_direction(
        original_promoter_mlp
    )
)

non_promoter_direction_scores = (
    project_onto_promoter_direction(
        original_non_promoter_mlp
    )
)

direction_class_summary = pd.DataFrame([
    {
        "class": "promoter",
        "number_of_sequences": len(
            promoter_direction_scores
        ),
        "mean_direction_score": (
            promoter_direction_scores.mean()
        ),
        "median_direction_score": (
            np.median(
                promoter_direction_scores
            )
        ),
        "std_direction_score": (
            promoter_direction_scores.std(
                ddof=1
            )
        ),
    },
    {
        "class": "non-promoter",
        "number_of_sequences": len(
            non_promoter_direction_scores
        ),
        "mean_direction_score": (
            non_promoter_direction_scores.mean()
        ),
        "median_direction_score": (
            np.median(
                non_promoter_direction_scores
            )
        ),
        "std_direction_score": (
            non_promoter_direction_scores.std(
                ddof=1
            )
        ),
    },
])

print(
    "\nOriginal sequence scores on the "
    "Layer-1 MLP promoter direction:"
)

display(direction_class_summary)


# Add the representation index and direction score
# to every intervention row.
gc_direction_df = (
    gc_intervention_scored_df.copy()
)

gc_direction_df[
    "representation_index"
] = (
    gc_direction_df[
        "variant_sequence"
    ].map(
        sequence_to_representation_index
    )
)

assert gc_direction_df[
    "representation_index"
].notna().all()

gc_direction_df[
    "representation_index"
] = (
    gc_direction_df[
        "representation_index"
    ].astype(int)
)

gc_direction_df[
    "mlp_promoter_direction_score"
] = (
    project_onto_promoter_direction(
        unique_layer1_mlp_representations[
            gc_direction_df[
                "representation_index"
            ].to_numpy()
        ]
    )
)


# Obtain the original direction score for each
# matched intervention block.
block_keys = [
    "source_class",
    "source_index",
    "mutation_level",
    "variant_seed",
]

original_direction_df = (
    gc_direction_df[
        gc_direction_df[
            "intervention_type"
        ] == "original"
    ][
        block_keys
        + [
            "mlp_promoter_direction_score"
        ]
    ]
    .rename(
        columns={
            "mlp_promoter_direction_score":
                "original_mlp_direction_score"
        }
    )
)

gc_direction_df = (
    gc_direction_df.merge(
        original_direction_df,
        on=block_keys,
        how="left",
        validate="many_to_one",
    )
)

gc_direction_df[
    "mlp_direction_change"
] = (
    gc_direction_df[
        "mlp_promoter_direction_score"
    ]
    - gc_direction_df[
        "original_mlp_direction_score"
    ]
)


# Summarize raw intervention effects.
direction_intervention_summary = (
    gc_direction_df
    .groupby(
        [
            "source_class",
            "mutation_level",
            "intervention_type",
        ]
    )
    .agg(
        number_of_rows=(
            "mlp_direction_change",
            "size",
        ),
        mean_direction_change=(
            "mlp_direction_change",
            "mean",
        ),
        median_direction_change=(
            "mlp_direction_change",
            "median",
        ),
        std_direction_change=(
            "mlp_direction_change",
            "std",
        ),
        fraction_positive_change=(
            "mlp_direction_change",
            lambda values: np.mean(
                values > 0
            ),
        ),
    )
    .reset_index()
)

print(
    "\nLayer-1 MLP direction changes:"
)

display(direction_intervention_summary)


# Put GC-up, GC-down, and neutral controls
# on the same matched row.
direction_matched_df = (
    gc_direction_df[
        gc_direction_df[
            "intervention_type"
        ].isin(
            [
                "gc_increased",
                "gc_decreased",
                "gc_neutral",
            ]
        )
    ]
    .pivot(
        index=block_keys,
        columns="intervention_type",
        values="mlp_direction_change",
    )
    .reset_index()
)

direction_matched_df.columns.name = None

direction_matched_df[
    "gc_increase_vs_neutral"
] = (
    direction_matched_df[
        "gc_increased"
    ]
    - direction_matched_df[
        "gc_neutral"
    ]
)

direction_matched_df[
    "gc_decrease_vs_neutral"
] = (
    direction_matched_df[
        "gc_decreased"
    ]
    - direction_matched_df[
        "gc_neutral"
    ]
)

direction_matched_df[
    "gc_increase_vs_decrease"
] = (
    direction_matched_df[
        "gc_increased"
    ]
    - direction_matched_df[
        "gc_decreased"
    ]
)


direction_contrast_summary = (
    direction_matched_df
    .groupby(
        [
            "source_class",
            "mutation_level",
        ]
    )
    .agg(
        number_of_blocks=(
            "source_index",
            "size",
        ),
        mean_gc_increase_vs_neutral=(
            "gc_increase_vs_neutral",
            "mean",
        ),
        mean_gc_decrease_vs_neutral=(
            "gc_decrease_vs_neutral",
            "mean",
        ),
        mean_gc_increase_vs_decrease=(
            "gc_increase_vs_decrease",
            "mean",
        ),
        fraction_gc_increase_above_neutral=(
            "gc_increase_vs_neutral",
            lambda values: np.mean(
                values > 0
            ),
        ),
        fraction_gc_decrease_below_neutral=(
            "gc_decrease_vs_neutral",
            lambda values: np.mean(
                values < 0
            ),
        ),
        fraction_gc_increase_above_decrease=(
            "gc_increase_vs_decrease",
            lambda values: np.mean(
                values > 0
            ),
        ),
    )
    .reset_index()
)

print(
    "\nMatched Layer-1 MLP direction contrasts:"
)

display(direction_contrast_summary)


# Correlate internal direction changes with
# downstream Layer-5 probe-logit changes.
non_original_direction_df = (
    gc_direction_df[
        gc_direction_df[
            "intervention_type"
        ] != "original"
    ]
)

from scipy.stats import spearmanr

direction_probe_correlation = spearmanr(
    non_original_direction_df[
        "mlp_direction_change"
    ],
    non_original_direction_df[
        "probe_logit_change"
    ],
)

print(
    "\nCorrelation between Layer-1 MLP direction "
    "change and Layer-5 probe-logit change:"
)

print(
    "Spearman rho:",
    direction_probe_correlation.statistic,
)

print(
    "p-value:",
    direction_probe_correlation.pvalue,
)


# Save results.
np.savez_compressed(
    NOTEBOOK6_DIR
    / "layer1_mlp_promoter_direction.npz",
    promoter_direction=(
        promoter_direction
    ),
    promoter_direction_unit=(
        promoter_direction_unit
    ),
    promoter_mean_vector=(
        promoter_mean_vector
    ),
    non_promoter_mean_vector=(
        non_promoter_mean_vector
    ),
)

direction_class_summary.to_csv(
    NOTEBOOK6_DIR
    / "layer1_mlp_direction_class_summary.csv",
    index=False,
)

gc_direction_df.to_csv(
    NOTEBOOK6_DIR
    / "gc_intervention_mlp_direction_scores.csv",
    index=False,
)

direction_matched_df.to_csv(
    NOTEBOOK6_DIR
    / "gc_intervention_mlp_direction_contrasts.csv",
    index=False,
)

direction_contrast_summary.to_csv(
    NOTEBOOK6_DIR
    / "gc_intervention_mlp_direction_summary.csv",
    index=False,
)

print(
    "\nSaved Layer-1 MLP promoter-direction "
    "analysis to:",
    NOTEBOOK6_DIR,
)

Original promoter MLP matrix: (189, 768)
Original non-promoter MLP matrix: (189, 768)

Promoter direction norm: 0.4571739

Original sequence scores on the Layer-1 MLP promoter direction:


,class,number_of_sequences,mean_direction_score,median_direction_score,std_direction_score
0,promoter,189,0.294587,0.263622,0.514750
1,non-promoter,189,-0.162587,-0.203536,0.260845



Layer-1 MLP direction changes:


,source_class,mutation_level,intervention_type,number_of_rows,mean_direction_change,median_direction_change,std_direction_change,fraction_positive_change
0,non-promoter,4,gc_decreased,945,-0.115005,-0.109527,0.110255,0.157672
1,non-promoter,4,gc_increased,945,0.169750,0.166107,0.116755,0.936508
2,non-promoter,4,gc_neutral,945,0.004266,0.002533,0.091022,0.508995
3,non-promoter,4,original,945,0.000000,0.000000,0.000000,0.000000
4,non-promoter,8,gc_decreased,945,-0.193574,-0.184575,0.172498,0.115344
5,non-promoter,8,gc_increased,945,0.384251,0.393859,0.185270,0.969312
6,non-promoter,8,gc_neutral,945,0.007166,0.005215,0.124241,0.516402
7,non-promoter,8,original,945,0.000000,0.000000,0.000000,0.000000
8,promoter,4,gc_decreased,945,-0.211227,-0.214669,0.138210,0.064550
9,promoter,4,gc_increased,945,0.209495,0.209852,0.129817,0.939683



Matched Layer-1 MLP direction contrasts:


,source_class,mutation_level,number_of_blocks,mean_gc_increase_vs_neutral,mean_gc_decrease_vs_neutral,mean_gc_increase_vs_decrease,fraction_gc_increase_above_neutral,fraction_gc_decrease_below_neutral,fraction_gc_increase_above_decrease
0,non-promoter,4,945,0.165484,-0.119271,0.284755,0.886772,0.805291,0.953439
1,non-promoter,8,945,0.377085,-0.200740,0.577825,0.957672,0.840212,0.957672
2,promoter,4,945,0.229763,-0.190959,0.420722,0.939683,0.876190,0.957672
3,promoter,8,945,0.496020,-0.334204,0.830224,0.986243,0.901587,0.971429



Correlation between Layer-1 MLP direction change and Layer-5 probe-logit change:
Spearman rho: 0.39012291483656775
p-value: 0.0

Saved Layer-1 MLP promoter-direction analysis to: /content/drive/MyDrive/algoverse/dnabert/outputs/gc_controlled_interventions


# Cell 9 — Cross-validated Layer-1 MLP promoter direction and GC intervention test

In [15]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

CV_SEED = 42
N_SPLITS = 5

original_mlp_matrix = np.concatenate(
    [
        original_promoter_mlp,
        original_non_promoter_mlp,
    ],
    axis=0,
)

original_labels = np.concatenate(
    [
        np.ones(
            len(original_promoter_mlp),
            dtype=int,
        ),
        np.zeros(
            len(original_non_promoter_mlp),
            dtype=int,
        ),
    ]
)

original_sequences = (
    original_promoter_sequences
    + original_non_promoter_sequences
)

original_class_names = np.array(
    ["promoter"] * len(
        original_promoter_sequences
    )
    + ["non-promoter"] * len(
        original_non_promoter_sequences
    )
)

cross_validated_direction_scores = np.empty(
    len(original_mlp_matrix),
    dtype=float,
)

fold_direction_vectors = []

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=CV_SEED,
)

cv_fold_rows = []

for fold_index, (
    train_indices,
    test_indices,
) in enumerate(
    cv.split(
        original_mlp_matrix,
        original_labels,
    )
):
    train_promoter_mean = (
        original_mlp_matrix[
            train_indices
        ][
            original_labels[
                train_indices
            ] == 1
        ].mean(axis=0)
    )

    train_non_promoter_mean = (
        original_mlp_matrix[
            train_indices
        ][
            original_labels[
                train_indices
            ] == 0
        ].mean(axis=0)
    )

    fold_direction = (
        train_promoter_mean
        - train_non_promoter_mean
    )

    fold_direction /= np.linalg.norm(
        fold_direction
    )

    fold_direction_vectors.append(
        fold_direction
    )

    fold_scores = (
        original_mlp_matrix[
            test_indices
        ]
        @ fold_direction
    )

    cross_validated_direction_scores[
        test_indices
    ] = fold_scores

    fold_auc = roc_auc_score(
        original_labels[test_indices],
        fold_scores,
    )

    cv_fold_rows.append({
        "fold": fold_index + 1,
        "number_of_train_sequences": len(
            train_indices
        ),
        "number_of_test_sequences": len(
            test_indices
        ),
        "test_roc_auc": fold_auc,
    })


cv_direction_fold_summary = pd.DataFrame(
    cv_fold_rows
)

overall_cv_auc = roc_auc_score(
    original_labels,
    cross_validated_direction_scores,
)

print("Cross-validated direction performance:")
display(cv_direction_fold_summary)

print(
    "\nOverall out-of-fold ROC-AUC:",
    overall_cv_auc,
)


# Assign each source sequence to the fold in which it was held out.
sequence_fold_lookup = {}

for fold_index, (
    train_indices,
    test_indices,
) in enumerate(
    cv.split(
        original_mlp_matrix,
        original_labels,
    )
):
    for test_index in test_indices:
        sequence_fold_lookup[
            (
                original_class_names[
                    test_index
                ],
                (
                    test_index
                    if original_labels[
                        test_index
                    ] == 1
                    else test_index
                    - len(
                        original_promoter_sequences
                    )
                ),
            )
        ] = fold_index


# Compute intervention projections using only the direction
# learned without that source sequence.
cv_gc_direction_df = (
    gc_intervention_scored_df.copy()
)

cv_gc_direction_scores = np.empty(
    len(cv_gc_direction_df),
    dtype=float,
)

for row_position, row in enumerate(
    cv_gc_direction_df.itertuples()
):
    fold_index = sequence_fold_lookup[
        (
            row.source_class,
            row.source_index,
        )
    ]

    representation_index = (
        sequence_to_representation_index[
            row.variant_sequence
        ]
    )

    representation = (
        unique_layer1_mlp_representations[
            representation_index
        ]
    )

    cv_gc_direction_scores[
        row_position
    ] = (
        representation
        @ fold_direction_vectors[
            fold_index
        ]
    )


cv_gc_direction_df[
    "cv_mlp_direction_score"
] = (
    cv_gc_direction_scores
)


block_keys = [
    "source_class",
    "source_index",
    "mutation_level",
    "variant_seed",
]

cv_original_direction_df = (
    cv_gc_direction_df[
        cv_gc_direction_df[
            "intervention_type"
        ] == "original"
    ][
        block_keys
        + ["cv_mlp_direction_score"]
    ]
    .rename(
        columns={
            "cv_mlp_direction_score":
                "cv_original_direction_score"
        }
    )
)

cv_gc_direction_df = (
    cv_gc_direction_df.merge(
        cv_original_direction_df,
        on=block_keys,
        how="left",
        validate="many_to_one",
    )
)

cv_gc_direction_df[
    "cv_mlp_direction_change"
] = (
    cv_gc_direction_df[
        "cv_mlp_direction_score"
    ]
    - cv_gc_direction_df[
        "cv_original_direction_score"
    ]
)


cv_direction_matched_df = (
    cv_gc_direction_df[
        cv_gc_direction_df[
            "intervention_type"
        ].isin(
            [
                "gc_increased",
                "gc_decreased",
                "gc_neutral",
            ]
        )
    ]
    .pivot(
        index=block_keys,
        columns="intervention_type",
        values="cv_mlp_direction_change",
    )
    .reset_index()
)

cv_direction_matched_df.columns.name = None

cv_direction_matched_df[
    "gc_increase_vs_neutral"
] = (
    cv_direction_matched_df[
        "gc_increased"
    ]
    - cv_direction_matched_df[
        "gc_neutral"
    ]
)

cv_direction_matched_df[
    "gc_decrease_vs_neutral"
] = (
    cv_direction_matched_df[
        "gc_decreased"
    ]
    - cv_direction_matched_df[
        "gc_neutral"
    ]
)

cv_direction_matched_df[
    "gc_increase_vs_decrease"
] = (
    cv_direction_matched_df[
        "gc_increased"
    ]
    - cv_direction_matched_df[
        "gc_decreased"
    ]
)


cv_source_level_direction_df = (
    cv_direction_matched_df
    .groupby(
        [
            "source_class",
            "source_index",
            "mutation_level",
        ]
    )[
        [
            "gc_increase_vs_neutral",
            "gc_decrease_vs_neutral",
            "gc_increase_vs_decrease",
        ]
    ]
    .mean()
    .reset_index()
)


N_BOOTSTRAPS = 5000
bootstrap_rng = np.random.default_rng(42)

cv_bootstrap_rows = []

for (
    source_class,
    mutation_level,
), group_df in (
    cv_source_level_direction_df
    .groupby(
        [
            "source_class",
            "mutation_level",
        ]
    )
):
    for contrast in [
        "gc_increase_vs_neutral",
        "gc_decrease_vs_neutral",
        "gc_increase_vs_decrease",
    ]:
        values = group_df[
            contrast
        ].to_numpy()

        bootstrap_means = np.empty(
            N_BOOTSTRAPS
        )

        for bootstrap_index in range(
            N_BOOTSTRAPS
        ):
            sampled_indices = (
                bootstrap_rng.integers(
                    0,
                    len(values),
                    size=len(values),
                )
            )

            bootstrap_means[
                bootstrap_index
            ] = values[
                sampled_indices
            ].mean()

        cv_bootstrap_rows.append({
            "source_class": source_class,
            "mutation_level": mutation_level,
            "contrast": contrast,
            "number_of_sources": len(values),
            "mean_effect": values.mean(),
            "median_effect": np.median(
                values
            ),
            "ci_2.5": np.percentile(
                bootstrap_means,
                2.5,
            ),
            "ci_97.5": np.percentile(
                bootstrap_means,
                97.5,
            ),
            "fraction_sources_positive": (
                np.mean(values > 0)
            ),
            "fraction_sources_negative": (
                np.mean(values < 0)
            ),
        })


cv_direction_bootstrap_summary = (
    pd.DataFrame(cv_bootstrap_rows)
)

print(
    "\nCross-validated GC effects in "
    "Layer-1 MLP promoter direction:"
)

display(
    cv_direction_bootstrap_summary.sort_values(
        [
            "source_class",
            "mutation_level",
            "contrast",
        ]
    )
)


cv_non_original_df = (
    cv_gc_direction_df[
        cv_gc_direction_df[
            "intervention_type"
        ] != "original"
    ]
)

cv_direction_probe_correlation = (
    spearmanr(
        cv_non_original_df[
            "cv_mlp_direction_change"
        ],
        cv_non_original_df[
            "probe_logit_change"
        ],
    )
)

print(
    "\nCross-validated internal/downstream correlation:"
)

print(
    "Spearman rho:",
    cv_direction_probe_correlation.statistic,
)

print(
    "p-value:",
    cv_direction_probe_correlation.pvalue,
)


cv_direction_fold_summary.to_csv(
    NOTEBOOK6_DIR
    / "layer1_mlp_direction_cross_validation_folds.csv",
    index=False,
)

cv_gc_direction_df.to_csv(
    NOTEBOOK6_DIR
    / "gc_intervention_cross_validated_direction_scores.csv",
    index=False,
)

cv_direction_bootstrap_summary.to_csv(
    NOTEBOOK6_DIR
    / "gc_intervention_cross_validated_direction_bootstrap.csv",
    index=False,
)

print(
    "\nSaved cross-validated direction analysis to:",
    NOTEBOOK6_DIR,
)

Cross-validated direction performance:


,fold,number_of_train_sequences,number_of_test_sequences,test_roc_auc
0,1,302,76,0.792244
1,2,302,76,0.800554
2,3,302,76,0.759695
3,4,303,75,0.741821
4,5,303,75,0.669986



Overall out-of-fold ROC-AUC: 0.7556339408191262

Cross-validated GC effects in Layer-1 MLP promoter direction:


,source_class,mutation_level,contrast,number_of_sources,mean_effect,median_effect,ci_2.5,ci_97.5,fraction_sources_positive,fraction_sources_negative
1,non-promoter,4,gc_decrease_vs_neutral,189,-0.118289,-0.119529,-0.130755,-0.105633,0.079365,0.920635
2,non-promoter,4,gc_increase_vs_decrease,189,0.283139,0.276941,0.261178,0.304503,0.957672,0.042328
0,non-promoter,4,gc_increase_vs_neutral,189,0.164850,0.158406,0.152704,0.176982,0.978836,0.021164
4,non-promoter,8,gc_decrease_vs_neutral,189,-0.198521,-0.199321,-0.219909,-0.177311,0.079365,0.920635
5,non-promoter,8,gc_increase_vs_decrease,189,0.574571,0.581885,0.535897,0.613320,0.957672,0.042328
3,non-promoter,8,gc_increase_vs_neutral,189,0.376050,0.389619,0.353596,0.398077,0.973545,0.026455
7,promoter,4,gc_decrease_vs_neutral,189,-0.190379,-0.209521,-0.206827,-0.173909,0.074074,0.925926
8,promoter,4,gc_increase_vs_decrease,189,0.417624,0.479574,0.389223,0.444797,0.957672,0.042328
6,promoter,4,gc_increase_vs_neutral,189,0.227245,0.237988,0.212442,0.242073,0.973545,0.026455
10,promoter,8,gc_decrease_vs_neutral,189,-0.331669,-0.365282,-0.361879,-0.299920,0.084656,0.915344



Cross-validated internal/downstream correlation:
Spearman rho: 0.3867541180330471
p-value: 0.0

Saved cross-validated direction analysis to: /content/drive/MyDrive/algoverse/dnabert/outputs/gc_controlled_interventions


# Cell 10 — Finalize Notebook 6 results and save a compact summary

In [16]:
import json
from pathlib import Path

# Extract the most important cross-validated results.
def get_cv_result(
    source_class,
    mutation_level,
    contrast,
):
    row = cv_direction_bootstrap_summary[
        (
            cv_direction_bootstrap_summary[
                "source_class"
            ] == source_class
        )
        & (
            cv_direction_bootstrap_summary[
                "mutation_level"
            ] == mutation_level
        )
        & (
            cv_direction_bootstrap_summary[
                "contrast"
            ] == contrast
        )
    ]

    if len(row) != 1:
        raise ValueError(
            "Expected exactly one matching result for "
            f"{source_class}, {mutation_level}, {contrast}."
        )

    return row.iloc[0]


summary_records = []

for source_class in [
    "promoter",
    "non-promoter",
]:
    for mutation_level in [4, 8]:
        for contrast in [
            "gc_increase_vs_neutral",
            "gc_decrease_vs_neutral",
            "gc_increase_vs_decrease",
        ]:
            result = get_cv_result(
                source_class,
                mutation_level,
                contrast,
            )

            summary_records.append({
                "source_class": source_class,
                "mutation_level": int(
                    mutation_level
                ),
                "contrast": contrast,
                "mean_effect": float(
                    result["mean_effect"]
                ),
                "median_effect": float(
                    result["median_effect"]
                ),
                "ci_2.5": float(
                    result["ci_2.5"]
                ),
                "ci_97.5": float(
                    result["ci_97.5"]
                ),
                "fraction_sources_positive": float(
                    result[
                        "fraction_sources_positive"
                    ]
                ),
                "fraction_sources_negative": float(
                    result[
                        "fraction_sources_negative"
                    ]
                ),
            })


final_cv_effect_df = pd.DataFrame(
    summary_records
)

print(
    "Final cross-validated Layer-1 MLP "
    "GC-effect summary:"
)

display(final_cv_effect_df)


# Calculate cross-validated dose scaling.
cv_dose_df = (
    cv_source_level_direction_df
    .pivot(
        index=[
            "source_class",
            "source_index",
        ],
        columns="mutation_level",
        values=[
            "gc_increase_vs_neutral",
            "gc_decrease_vs_neutral",
            "gc_increase_vs_decrease",
        ],
    )
)

dose_rows = []

for source_class in [
    "promoter",
    "non-promoter",
]:
    class_df = cv_dose_df.loc[
        source_class
    ]

    for contrast in [
        "gc_increase_vs_neutral",
        "gc_decrease_vs_neutral",
        "gc_increase_vs_decrease",
    ]:
        effect_4 = class_df[
            (contrast, 4)
        ].to_numpy()

        effect_8 = class_df[
            (contrast, 8)
        ].to_numpy()

        difference = effect_8 - effect_4

        dose_rows.append({
            "source_class": source_class,
            "contrast": contrast,
            "mean_effect_4": float(
                effect_4.mean()
            ),
            "mean_effect_8": float(
                effect_8.mean()
            ),
            "mean_8_minus_4": float(
                difference.mean()
            ),
            "median_8_minus_4": float(
                np.median(difference)
            ),
            "fraction_effect_larger_at_8": float(
                np.mean(effect_8 > effect_4)
            ),
        })


cv_dose_summary_df = pd.DataFrame(
    dose_rows
)

print("\nCross-validated dose-response summary:")
display(cv_dose_summary_df)


notebook6_summary = {
    "notebook": "06_gc_controlled_interventions",
    "model": MODEL_NAME,
    "dataset": "GUE prom_core_all pilot test cohorts",
    "source_sequences": {
        "promoter": 189,
        "non_promoter": 189,
    },
    "sequence_length_nt": 70,
    "mutation_levels": [4, 8],
    "variant_seeds_per_source": 5,
    "total_intervention_rows": int(
        len(gc_intervention_scored_df)
    ),
    "unique_variant_sequences": int(
        len(unique_sequences)
    ),
    "layer5_probe": {
        "description": (
            "Fixed diagnostic logistic probe trained on "
            "mean-pooled Layer-5 DNABERT-2 representations."
        ),
        "interpretation": (
            "Probe scores measure promoter-associated "
            "evidence under the diagnostic readout; they "
            "are not native DNABERT-2 predictions."
        ),
    },
    "layer1_mlp_direction": {
        "construction": (
            "Difference between mean promoter and "
            "non-promoter pooled Layer-1 MLP outputs."
        ),
        "cross_validation": (
            "Five-fold stratified cross-validation; each "
            "source and all of its intervention variants "
            "were evaluated using a direction learned "
            "without that source."
        ),
        "out_of_fold_roc_auc": float(
            overall_cv_auc
        ),
    },
    "internal_downstream_association": {
        "spearman_rho": float(
            cv_direction_probe_correlation.statistic
        ),
        "p_value": float(
            cv_direction_probe_correlation.pvalue
        ),
    },
    "main_supported_claim": (
        "Controlled GC-increasing substitutions move "
        "Layer-1 MLP activations toward a held-out "
        "promoter-associated direction, while "
        "GC-decreasing substitutions move activations "
        "away from that direction. Effects exceed "
        "mutation-count-matched GC-neutral controls and "
        "increase with intervention magnitude."
    ),
    "limitations": [
        (
            "The model is a pretrained DNABERT-2 "
            "backbone with a diagnostic probe, not a "
            "fine-tuned native promoter classifier."
        ),
        (
            "Base substitutions alter local sequence "
            "context, motifs, dinucleotides, and BPE "
            "tokenization in addition to GC content."
        ),
        (
            "GC-neutral controls match mutation count but "
            "not the exact mutated positions or source-base "
            "classes of every directional intervention."
        ),
        (
            "The results establish effects on internal "
            "representations and the downstream probe, but "
            "do not establish that the Layer-1 MLP fully "
            "mediates the downstream effect."
        ),
        (
            "The source cohort contains selected recognized "
            "promoters and sampled non-promoters, limiting "
            "population-level generalization."
        ),
    ],
}


final_cv_effect_df.to_csv(
    NOTEBOOK6_DIR
    / "notebook6_final_cv_effects.csv",
    index=False,
)

cv_dose_summary_df.to_csv(
    NOTEBOOK6_DIR
    / "notebook6_final_cv_dose_response.csv",
    index=False,
)

with open(
    NOTEBOOK6_DIR
    / "notebook6_summary.json",
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        notebook6_summary,
        output_file,
        indent=2,
    )


summary_markdown = f"""
# Notebook 6 — GC-Controlled Interventions

## Main result

Controlled GC-altering substitutions causally moved pooled
Layer-1 MLP activations along a held-out promoter-associated
direction.

The five-fold out-of-fold ROC-AUC of the Layer-1 promoter
direction was **{overall_cv_auc:.3f}**.

For promoter source sequences:

- 4-mutation GC-up versus neutral:
  **+{get_cv_result("promoter", 4, "gc_increase_vs_neutral")["mean_effect"]:.3f}**
- 8-mutation GC-up versus neutral:
  **+{get_cv_result("promoter", 8, "gc_increase_vs_neutral")["mean_effect"]:.3f}**
- 4-mutation GC-down versus neutral:
  **{get_cv_result("promoter", 4, "gc_decrease_vs_neutral")["mean_effect"]:.3f}**
- 8-mutation GC-down versus neutral:
  **{get_cv_result("promoter", 8, "gc_decrease_vs_neutral")["mean_effect"]:.3f}**

The cross-validated internal Layer-1 MLP change correlated
with the downstream Layer-5 diagnostic-probe change at
Spearman rho = **{cv_direction_probe_correlation.statistic:.3f}**.

## Interpretation

GC content contributes causally to a promoter-associated
representation present in the Layer-1 MLP output. Increasing
GC shifts this representation toward the promoter direction,
whereas decreasing GC shifts it away. The directional
separation becomes larger at eight mutations than at four.

## Claim boundary

These results concern a fixed diagnostic promoter probe over
a pretrained DNABERT-2 backbone. They do not establish a
native DNABERT-2 promoter prediction mechanism or complete
causal mediation through the Layer-1 MLP.
""".strip()


with open(
    NOTEBOOK6_DIR
    / "NOTEBOOK6_RESULTS.md",
    "w",
    encoding="utf-8",
) as output_file:
    output_file.write(
        summary_markdown
    )


print(
    "\nSaved final files:"
)

for filename in [
    "notebook6_final_cv_effects.csv",
    "notebook6_final_cv_dose_response.csv",
    "notebook6_summary.json",
    "NOTEBOOK6_RESULTS.md",
]:
    print(
        NOTEBOOK6_DIR / filename
    )

print(
    "\nNotebook 6 completed successfully."
)

Final cross-validated Layer-1 MLP GC-effect summary:


,source_class,mutation_level,contrast,mean_effect,median_effect,ci_2.5,ci_97.5,fraction_sources_positive,fraction_sources_negative
0,promoter,4,gc_increase_vs_neutral,0.227245,0.237988,0.212442,0.242073,0.973545,0.026455
1,promoter,4,gc_decrease_vs_neutral,-0.190379,-0.209521,-0.206827,-0.173909,0.074074,0.925926
2,promoter,4,gc_increase_vs_decrease,0.417624,0.479574,0.389223,0.444797,0.957672,0.042328
3,promoter,8,gc_increase_vs_neutral,0.491745,0.526680,0.465438,0.517622,0.994709,0.005291
4,promoter,8,gc_decrease_vs_neutral,-0.331669,-0.365282,-0.361879,-0.299920,0.084656,0.915344
5,promoter,8,gc_increase_vs_decrease,0.823414,0.936927,0.771941,0.874741,0.978836,0.021164
6,non-promoter,4,gc_increase_vs_neutral,0.164850,0.158406,0.152704,0.176982,0.978836,0.021164
7,non-promoter,4,gc_decrease_vs_neutral,-0.118289,-0.119529,-0.130755,-0.105633,0.079365,0.920635
8,non-promoter,4,gc_increase_vs_decrease,0.283139,0.276941,0.261178,0.304503,0.957672,0.042328
9,non-promoter,8,gc_increase_vs_neutral,0.376050,0.389619,0.353596,0.398077,0.973545,0.026455



Cross-validated dose-response summary:


,source_class,contrast,mean_effect_4,mean_effect_8,mean_8_minus_4,median_8_minus_4,fraction_effect_larger_at_8
0,promoter,gc_increase_vs_neutral,0.227245,0.491745,0.264500,0.277935,0.984127
1,promoter,gc_decrease_vs_neutral,-0.190379,-0.331669,-0.141290,-0.146096,0.174603
2,promoter,gc_increase_vs_decrease,0.417624,0.823414,0.405790,0.441460,0.957672
3,non-promoter,gc_increase_vs_neutral,0.164850,0.376050,0.211201,0.216469,0.952381
4,non-promoter,gc_decrease_vs_neutral,-0.118289,-0.198521,-0.080232,-0.068262,0.201058
5,non-promoter,gc_increase_vs_decrease,0.283139,0.574571,0.291432,0.299288,0.968254



Saved final files:
/content/drive/MyDrive/algoverse/dnabert/outputs/gc_controlled_interventions/notebook6_final_cv_effects.csv
/content/drive/MyDrive/algoverse/dnabert/outputs/gc_controlled_interventions/notebook6_final_cv_dose_response.csv
/content/drive/MyDrive/algoverse/dnabert/outputs/gc_controlled_interventions/notebook6_summary.json
/content/drive/MyDrive/algoverse/dnabert/outputs/gc_controlled_interventions/NOTEBOOK6_RESULTS.md

Notebook 6 completed successfully.
